# Stage 1 — Non-Instruction Fine-Tuning (Healthcare FAQ Assistant)

**Goal:** adapt a base LLM to *healthcare language and terminology* by continuing pre-training on raw domain text **before** instruction tuning.

Pipeline: **Base Model → [Stage 1: Non-Instruction FT] → Stage 2: SFT → Stage 3: DPO**

This notebook:
1. Loads raw domain text (`data/non_instruction_data.txt`)
2. Cleans & chunks it
3. Loads the base model with Unsloth + 4-bit (QLoRA)
4. Applies LoRA (incl. embeddings for continued pre-training)
5. Trains on raw text
6. Saves the adapter + a merged model for Stage 2
7. Tests the model after non-instruction fine-tuning

> ⚠️ Educational project — the assistant gives general health information only and is not medical advice.

## 0. Install dependencies (Colab)

In [ ]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes.
%%capture
!pip install -q unsloth
!pip install -q --no-deps "trl<0.12" peft accelerate bitsandbytes

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [ ]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

## 1. Select base model

In [ ]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

## 2. Paths

In [ ]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = False                 # set True to upload after training
HF_USERNAME  = "your-hf-username"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

## 3. Load the base model with Unsloth (4-bit / QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,
    max_seq_length = max_seq_length,
    dtype          = None,        # auto: bf16 on Ampere+, else fp16
    load_in_4bit   = True,        # QLoRA
)
print('Base model loaded.')

## 4. Load, clean and chunk the raw domain text

For continued pre-training we feed plain text. We split the file into paragraphs, drop empties, and append the EOS token so the model learns where passages end.

In [ ]:
import os
from datasets import Dataset

raw_path = os.path.join(DATA_DIR, "non_instruction_data.txt")
with open(raw_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Clean + chunk: one training example per paragraph
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
paragraphs = [" ".join(p.split()) for p in paragraphs]   # collapse whitespace
print(f"Loaded {len(paragraphs)} paragraphs")

EOS = tokenizer.eos_token
texts = [p + EOS for p in paragraphs]
dataset = Dataset.from_dict({"text": texts})
print(dataset)
print("\nExample:\n", dataset[0]["text"][:300])

## 5. Apply LoRA

For continued pre-training we also train the `embed_tokens` and `lm_head` so the model can better absorb new domain vocabulary (this is the Unsloth continued-pretraining recipe).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj",
                      "embed_tokens","lm_head"],   # extra modules for continued pre-training
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)
print('LoRA adapters attached.')

## 6. Train on the raw domain text

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size 8
        warmup_steps = 5,
        num_train_epochs = 2,              # small dataset -> a couple of passes
        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,    # lower LR for embeddings
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage1_logs"),
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
trainer_stats

## 7. Save the adapter and a merged model (input to Stage 2)

In [ ]:
STAGE1_ADAPTER = os.path.join(OUTPUT_DIR, "stage1_non_instruction")
STAGE1_MERGED  = os.path.join(OUTPUT_DIR, "stage1_merged")

# LoRA adapter (small)
model.save_pretrained(STAGE1_ADAPTER)
tokenizer.save_pretrained(STAGE1_ADAPTER)
print("Saved adapter ->", STAGE1_ADAPTER)

# Merged 16-bit model so Stage 2 can load it as a clean base
model.save_pretrained_merged(STAGE1_MERGED, tokenizer, save_method="merged_16bit")
print("Saved merged model ->", STAGE1_MERGED)

### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage1"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage1-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage1-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage1-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 8. Test the model after non-instruction fine-tuning

This stage teaches *domain language*, not Q&A behaviour yet — so we test text **completion**. The model should continue healthcare text in a domain-appropriate style.

In [ ]:
FastLanguageModel.for_inference(model)

# 10 completion prompts — each maps to the matching question # in
# reports/base_model_evaluation.md (rephrased as sentence-starters,
# since this stage produces a text-completion / non-instruction model).
prompts = [
    "An employee who has the flu and needs to apply for sick leave should",              # 1
    "In adults, a body temperature is considered a fever when it reaches",               # 2
    "For a common cold, taking antibiotics is",                                          # 3
    "A person taking blood pressure medication who feels fine should",                   # 4
    "If a baby under three months old has a fever, the parents should",                  # 5
    "The warning signs of a stroke include",                                             # 6
    "When a headache will not go away, the amount of paracetamol a person can take is",  # 7
    "A safe and effective way to lose weight is",                                        # 8
    "To manage type 2 diabetes, a person can",                                           # 9
    "This Healthcare FAQ Assistant can help with",                                       # 10
]
for p in prompts:
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=60, use_cache=True, do_sample=False)
    print("PROMPT:", p)
    print("CONT :", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 80)

## Done — Stage 1 complete ✅

Next: open **`instruction_finetuning.ipynb`** and set `RESUME_FROM_STAGE1 = True` to continue from `outputs/stage1_merged`.